# PRIVATE Korean voice quality lab — OpenVoice V2 + MeloTTS

The earlier bundle was rejected for metallic/robotic speech and weak identity.
This notebook does NOT approve it or any new bundle automatically.

Keep this notebook private. Enable Internet and a free Kaggle GPU (or free Colab).
Use session upload ONLY, not a Kaggle Dataset. Raw audio, selected speech,
embeddings, metrics, and comparisons live under `/tmp`. All are PRIVATE.
Never save/publish notebook versions with preview, widget or download outputs.
Clear ALL outputs before exporting notebook source; download assets then end the session.

Workflow: helper/config cell → optional setup → upload → select/listen →
extract → compare/listen → explicitly approve a mode → export/download.
Rerun selection/extraction/generation independently in the same live session;
no full environment reinstall is needed. A new session loses `/tmp` assets and
needs setup/upload again. The old rejected bundle is not overwritten.

OpenVoice transfers tone color, NOT reference rhythm/emotion. Flat MeloTTS
pacing cannot be fixed merely by collecting more reference audio.


In [ ]:
# Embedded PUBLIC helper source: works before repository changes are pushed.
import os
import sys
import subprocess
from pathlib import Path
SCRIPTS = Path('/tmp/manhua-lens/voice_server')
SCRIPTS.mkdir(parents=True, exist_ok=True)
PUBLIC_SOURCES = {'voice_assets.py': '"""Private phrase-cache format shared by preparation and playback."""\nimport hashlib\nimport io\nimport json\nimport re\nimport unicodedata\nimport wave\nimport zipfile\nfrom pathlib import Path\n\nMAX_WAV = 32 * 1024 * 1024\n\n\ndef normalize_text(text):\n    if not isinstance(text, str):\n        raise ValueError("text must be a string")\n    text = unicodedata.normalize("NFC", text).strip()\n    if not text or len(text) > 400:\n        raise ValueError("text must contain 1–400 characters")\n    return text\n\n\ndef cache_name(text):\n    return hashlib.sha256(normalize_text(text).encode("utf-8")).hexdigest() + ".wav"\n\n\ndef validate_wav(data):\n    if len(data) > MAX_WAV:\n        raise ValueError("WAV is too large")\n    try:\n        with wave.open(io.BytesIO(data), "rb") as audio:\n            if audio.getnframes() == 0 or audio.getnchannels() not in (1, 2):\n                raise ValueError("Empty or unsupported WAV")\n            expected = audio.getnframes() * audio.getnchannels() * audio.getsampwidth()\n            if len(audio.readframes(audio.getnframes())) != expected:\n                raise ValueError("Truncated WAV")\n    except (wave.Error, EOFError) as exc:\n        raise ValueError("Expected a PCM WAV") from exc\n    return data\n\n\ndef read_cached(root, text):\n    path = Path(root) / "cache" / cache_name(text)\n    if not path.is_file():\n        return None\n    if path.stat().st_size > MAX_WAV:\n        raise ValueError("WAV is too large")\n    return validate_wav(path.read_bytes())\n\n\ndef import_bundle(archive, destination):\n    """Import only explicit asset names; never extract arbitrary archive paths."""\n    destination = Path(destination)\n    with zipfile.ZipFile(archive) as bundle:\n        infos = bundle.infolist()\n        if len(infos) > 10002 or sum(i.file_size for i in infos) > 512 * 1024 * 1024:\n            raise ValueError("Bundle exceeds the 512 MiB / 10,000 phrase limit")\n        names = [i.filename for i in infos]\n        if len(names) != len(set(names)) or "manifest.json" not in names:\n            raise ValueError("Missing manifest or duplicate bundle entries")\n        for info in infos:\n            name = info.filename\n            if name not in ("manifest.json", "target_se.pth") and not re.fullmatch(r"cache/[0-9a-f]{64}\\.wav", name):\n                raise ValueError("Unexpected bundle entry: " + name)\n            if info.file_size > (MAX_WAV if name.endswith(".wav") else 1024 * 1024):\n                raise ValueError("Oversized bundle entry: " + name)\n        manifest = json.loads(bundle.read("manifest.json"))\n        if not isinstance(manifest, dict) or manifest.get("format") != "manhua-lens-private-v1":\n            raise ValueError("Unsupported private bundle format")\n        for name in names:\n            if name.endswith(".wav"):\n                validate_wav(bundle.read(name))\n        destination.mkdir(parents=True, exist_ok=True)\n        for name in names:\n            target = destination / name\n            if destination.is_symlink() or target.is_symlink() or target.parent.is_symlink():\n                raise ValueError("Refusing a symlink in the private asset directory")\n            target.parent.mkdir(parents=True, exist_ok=True)\n            target.write_bytes(bundle.read(name))\n\n\nif __name__ == "__main__":\n    import argparse\n    parser = argparse.ArgumentParser(description="Import your own PRIVATE Kaggle bundle")\n    parser.add_argument("archive", type=Path)\n    parser.add_argument("destination", type=Path)\n    args = parser.parse_args()\n    import_bundle(args.archive, args.destination)\n    print("Private assets imported. Do not commit or share this directory.")\n', 'engine.py': '"""Optional full OpenVoice V2 + MeloTTS Korean inference."""\nimport tempfile\nimport threading\nfrom pathlib import Path\n\nQUALITY_MODES = {\n    "natural": {"speed": 1.0, "sdp_ratio": 0.2, "noise_scale": 0.6, "noise_scale_w": 0.8, "tau": 0.3},\n    # Experimental alternative, NOT a guaranteed similarity improvement.\n    "similarity": {"speed": 0.98, "sdp_ratio": 0.15, "noise_scale": 0.5, "noise_scale_w": 0.7, "tau": 0.25},\n}\n\n\nclass KoreanVoice:\n    def __init__(self, checkpoints, embedding, device=None):\n        import torch\n        from melo.api import TTS\n        from openvoice.api import ToneColorConverter\n        from melo.text import korean\n        from korean_frontend import create_phonemizer\n\n        korean.g2p_kr = create_phonemizer()\n\n        self.lock = threading.Lock()\n        self.device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")\n        root = Path(checkpoints)\n        self.converter = ToneColorConverter(str(root / "converter/config.json"), device=self.device)\n        self.converter.load_ckpt(str(root / "converter/checkpoint.pth"))\n        if self.converter.version != "v2":\n            raise ValueError("Expected OpenVoice V2 converter checkpoints")\n        self.target = torch.load(embedding, map_location=self.device, weights_only=True)\n        if not isinstance(self.target, torch.Tensor) or self.target.ndim != 3 or not torch.isfinite(self.target).all():\n            raise ValueError("Invalid target speaker embedding")\n        self.model = TTS(language="KR", device=self.device)\n        speakers = self.model.hps.data.spk2id\n        if "KR" not in speakers:\n            raise RuntimeError("Expected the MeloTTS Korean KR speaker")\n        self.speaker = speakers["KR"]\n        self.source_path = root / "base_speakers/ses/kr.pth"\n        self.source = torch.load(self.source_path, map_location=self.device, weights_only=True)\n        if not isinstance(self.source, torch.Tensor) or not torch.isfinite(self.source).all():\n            raise ValueError("Invalid Korean source embedding")\n        if self.target.shape != self.source.shape:\n            raise ValueError("Target embedding does not match OpenVoice V2")\n\n    def synthesize(self, text, mode="natural", diagnostics=None, seed=5140):\n        import torch\n\n        settings = QUALITY_MODES[mode]\n        if not self.lock.acquire(blocking=False):\n            raise RuntimeError("Voice service is busy")\n        try:\n            with tempfile.TemporaryDirectory(prefix="mhl-tts-") as tmp:\n                source, output = Path(tmp) / "source.wav", Path(tmp) / "cloned.wav"\n                # Seed locally so comparisons use repeatable Melo and conversion noise.\n                with torch.random.fork_rng(devices=[torch.device(self.device)] if "cuda" in self.device else []):\n                    torch.manual_seed(seed)\n                    self.model.tts_to_file(text, self.speaker, str(source), quiet=True,\n                                          **{k: v for k, v in settings.items() if k != "tau"})\n                    torch.manual_seed(seed)\n                    self.converter.convert(audio_src_path=str(source), src_se=self.source,\n                                           tgt_se=self.target, output_path=str(output),\n                                           tau=settings["tau"], message="@ManhuaLens")\n                    if diagnostics is not None:\n                        import shutil\n                        diagnostics = Path(diagnostics)\n                        diagnostics.mkdir(parents=True, exist_ok=True)\n                        shutil.copyfile(source, diagnostics / "base.wav")\n                        shutil.copyfile(output, diagnostics / "cloned.wav")\n                        # Source->source conversion isolates converter artifacts from identity transfer.\n                        torch.manual_seed(seed)\n                        self.converter.convert(audio_src_path=str(source), src_se=self.source,\n                                               tgt_se=self.source, output_path=str(diagnostics / "self.wav"),\n                                               tau=settings["tau"], message="@ManhuaLens")\n                return output.read_bytes()\n        finally:\n            self.lock.release()\n', 'korean_frontend.py': '"""Use prebuilt python-mecab-ko on Windows as well as Linux."""\n\n\ndef create_phonemizer():\n    from g2pkk import G2p\n    from mecab import MeCab\n\n    class PortableG2p(G2p):\n        def check_mecab(self):\n            # Dependencies are installed explicitly during setup. Never invoke\n            # g2pkk\'s implicit Windows `pip install eunjeon` / compiler path.\n            pass\n\n        def get_mecab(self):\n            return MeCab()\n\n    return PortableG2p()\n', 'setup_runtime.py': '"""Reproducible model setup inside a Python 3.10 venv (Windows or notebook)."""\nimport argparse\nimport json\nimport shutil\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nOPENVOICE_REV = "74a1d147b17a8c3092dd5430504bd83ef6c7eb23"\nMELO_REV = "209145371cff8fc3bd60d7be902ea69cbdb7965a"\nCHECKPOINT_REV = "f36e7edfe1684461a8343844af60babc2efbb727"\nHERE = Path(__file__).resolve().parent\n\n\ndef run(*args):\n    subprocess.run([str(a) for a in args], check=True)\n\n\ndef download_checkpoints(destination):\n    from huggingface_hub import hf_hub_download\n\n    destination = Path(destination)\n    # Only the converter and Korean base speaker are needed for this pipeline.\n    files = ["converter/config.json", "converter/checkpoint.pth", "base_speakers/ses/kr.pth"]\n    if all((destination / name).is_file() for name in files):\n        return\n    for name in files:\n        cached = hf_hub_download("myshell-ai/OpenVoiceV2", name, revision=CHECKPOINT_REV)\n        target = destination / name\n        target.parent.mkdir(parents=True, exist_ok=True)\n        partial = target.with_suffix(target.suffix + ".partial")\n        shutil.copyfile(cached, partial)\n        partial.replace(target)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--torch-index", choices=["cpu", "cu121"], default="cpu")\n    parser.add_argument("--checkpoints", type=Path, default=HERE / "OpenVoice/checkpoints_v2")\n    parser.add_argument("--extraction-only", action="store_true",\n                        help="Add official VAD extraction imports to an existing notebook runtime only")\n    args = parser.parse_args()\n    if sys.version_info[:2] != (3, 10) or sys.prefix == sys.base_prefix:\n        raise SystemExit("Run this with Python 3.10 inside an isolated venv. See README.md.")\n    pip = [sys.executable, "-m", "pip"]\n    if args.extraction_only:\n        # Whisper\'s setup imports pkg_resources; newer build-isolation setuptools\n        # dropped it. Use the known toolchain already used by the model runtime.\n        run(*pip, "install", "pip==24.3.1", "setuptools==69.5.1", "wheel==0.45.1")\n        run(*pip, "install", "--no-build-isolation", "-r", HERE / "requirements-extraction.txt",\n            "-c", HERE / "requirements-models.txt")\n        run(sys.executable, "-c", "from openvoice import se_extractor; print(\'Official extraction imports OK\')")\n        (Path(sys.prefix) / "manhua-extraction-ready.txt").write_text("official-vad-v1", encoding="utf-8")\n        return\n    run(*pip, "install", "pip==24.3.1", "setuptools==69.5.1", "wheel==0.45.1")\n    run(*pip, "install", "torch==2.5.1", "torchaudio==2.5.1", "--index-url",\n        "https://download.pytorch.org/whl/" + args.torch_index)\n    run(*pip, "install", "-r", HERE / "requirements-models.txt", "-r", HERE / "requirements.txt")\n    # Install audited inference dependencies explicitly, avoiding old OpenVoice ASR\n    # pins and unused Gradio servers. No faster-whisper/Whisper extraction is used.\n    for repo, revision in [("OpenVoice", OPENVOICE_REV), ("MeloTTS", MELO_REV)]:\n        run(*pip, "install", "--no-deps", "--no-build-isolation",\n            f"https://github.com/myshell-ai/{repo}/archive/{revision}.zip")\n    run(sys.executable, "-m", "nltk.downloader", "-d", Path(sys.prefix) / "nltk_data",\n        "cmudict", "averaged_perceptron_tagger", "punkt")\n    # Melo\'s eager Japanese cleaner imports require the dictionary even for KR.\n    run(sys.executable, "-m", "unidic", "download")\n    run(sys.executable, "-c", "import sys; sys.path.insert(0, " + repr(str(HERE)) + "); from korean_frontend import create_phonemizer; assert create_phonemizer()(\'안녕하세요\'); print(\'Korean phonemizer OK\')")\n    download_checkpoints(args.checkpoints)\n    run(sys.executable, "-c", "from melo.api import TTS; from openvoice.api import ToneColorConverter; print(\'Model imports OK\')")\n    marker = {"openvoice": OPENVOICE_REV, "melo": MELO_REV, "checkpoints": CHECKPOINT_REV,\n              "torch": "2.5.1", "index": args.torch_index}\n    (Path(sys.prefix) / "manhua-ready.json").write_text(json.dumps(marker), encoding="utf-8")\n    print("Runtime installed. Model weights/tokenizers may download on first inference.")\n\n\nif __name__ == "__main__":\n    main()\n', 'prepare_voice.py': '"""Staged PRIVATE reference selection, official VAD extraction, comparison and export."""\nimport argparse\nimport hashlib\nimport json\nimport tempfile\nimport zipfile\nfrom pathlib import Path\n\nfrom voice_assets import cache_name, normalize_text, validate_wav\nfrom setup_runtime import OPENVOICE_REV, MELO_REV, CHECKPOINT_REV\n\nTEST_TEXT = \'안녕하세요. 오늘도 한국어 공부를 시작해 볼까요?\'\nCOMPARISON_TEXTS = [\'안녕하세요.\', \'오늘은 날씨가 좋아요.\', \'한국어 공부를 시작해 볼까요?\']\n\n\ndef pinned_vad(audio, **kwargs):\n    from whisper_timestamped.transcribe import get_vad_segments\n    # Pin the public Silero model tag rather than tracking changing master code.\n    kwargs[\'method\'] = \'silero:v5.1\'\n    return get_vad_segments(audio, **kwargs)\n\n\ndef select_reference(reference, work_dir, target_seconds=25):\n    import librosa\n    import numpy as np\n    import soundfile as sf\n    import torch\n    from reference_selection import select_segments, safe_normalize\n\n    work_dir = Path(work_dir)\n    work_dir.mkdir(parents=True, exist_ok=True)\n    info = sf.info(reference)\n    if not 20 <= info.duration <= 600:\n        raise ValueError(\'Supply 20–600s of clean speech; 20–30s will be selected\')\n    original, sr = sf.read(reference, dtype=\'float32\', always_2d=True)\n    if not np.isfinite(original).all():\n        raise ValueError(\'Reference contains invalid samples\')\n    # Detect clipping before resampling/gain can hide it. Channels must not cancel.\n    clipped = np.max(np.abs(original), axis=1) >= .995\n    mono = original.mean(axis=1)\n    if np.sqrt(np.mean(mono ** 2)) < .25 * np.sqrt(np.mean(original ** 2)):\n        raise ValueError(\'Stereo channels cancel; choose the clean microphone channel manually\')\n    mono = librosa.resample(mono, orig_sr=sr, target_sr=16000)\n    # Preserve clipping evidence on the VAD grid for acoustic screening.\n    positions = np.minimum((np.arange(len(mono)) * sr / 16000).astype(int), len(clipped) - 1)\n    clipping_mask = clipped[positions].copy()\n    # Downsampling must not hide individual clipped input samples. Keep evidence\n    # separate so screening never injects new peaks into the selected waveform.\n    clipped_positions = np.minimum((np.flatnonzero(clipped) * 16000 / sr).astype(int), len(mono) - 1)\n    clipping_mask[clipped_positions] = True\n    vad_audio, _ = safe_normalize(mono)\n    spans = pinned_vad(torch.from_numpy(vad_audio), output_sample=True,\n                       min_speech_duration=.3, min_silence_duration=.25, dilatation=.05)\n\n    def pitch_spread(chunk, rate):\n        f0, voiced, _ = librosa.pyin(chunk, fmin=65, fmax=500, sr=rate,\n                                    frame_length=1024, hop_length=256)\n        pitches = f0[voiced & np.isfinite(f0)]\n        if len(pitches) < 8:\n            return None\n        return float(12 * np.log2(np.percentile(pitches, 90) / np.percentile(pitches, 10)))\n\n    selected, report = select_segments(mono, 16000,\n        [(s[\'start\'], s[\'end\']) for s in spans], target_seconds, pitch_spread, clipping_mask)\n    selected, gain = safe_normalize(selected)\n    report.update(gain, input_seconds=info.duration, input_sample_rate=sr,\n                  selected_sample_rate=16000, vad=\'silero:v5.1\', rms_target_dbfs=-22,\n                  peak_ceiling_dbfs=-3, normalization=\'mono, resample, DC removal, constant gain; no denoising\')\n    sf.write(work_dir / \'selected_reference.wav\', selected, 16000, subtype=\'PCM_16\')\n    (work_dir / \'selection.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    print(f"Selected {report[\'selected_speech_seconds\']:.1f}s. LISTEN and confirm one calm speaker before extraction.")\n\n\ndef extract_embedding(checkpoints, work_dir, device=None):\n    import torch\n    from openvoice import se_extractor\n    from openvoice.api import ToneColorConverter\n\n    root, work_dir = Path(checkpoints), Path(work_dir)\n    selected = work_dir / \'selected_reference.wav\'\n    if not selected.is_file():\n        raise ValueError(\'Run reference selection first\')\n    device = device or (\'cuda:0\' if torch.cuda.is_available() else \'cpu\')\n    converter = ToneColorConverter(str(root / \'converter/config.json\'), device=device, enable_watermark=False)\n    converter.load_ckpt(str(root / \'converter/checkpoint.pth\'))\n    if converter.version != \'v2\':\n        raise ValueError(\'Expected OpenVoice V2\')\n    # Official get_se remains authoritative for VAD splitting and extraction.\n    # Only pin its VAD helper; never bypass get_se or reuse an old processed cache.\n    previous = se_extractor.get_vad_segments\n    se_extractor.get_vad_segments = pinned_vad\n    try:\n        with tempfile.TemporaryDirectory(prefix=\'official-vad-\', dir=work_dir) as processed:\n            target, _ = se_extractor.get_se(str(selected), converter, target_dir=processed, vad=True)\n    finally:\n        se_extractor.get_vad_segments = previous\n    source_path = root / \'base_speakers/ses/kr.pth\'\n    source = torch.load(source_path, map_location=device, weights_only=True)\n    if not isinstance(target, torch.Tensor) or target.shape != source.shape or not torch.isfinite(target).all():\n        raise ValueError(\'Invalid target embedding or mismatch with kr.pth\')\n    if not torch.isfinite(source).all():\n        raise ValueError(\'Invalid Korean source embedding\')\n    torch.save(target.detach().cpu(), work_dir / \'target_se.pth\')\n    report = {\'method\': \'openvoice.se_extractor.get_se(vad=True)\', \'vad\': \'silero:v5.1\',\n              \'source_embedding\': \'base_speakers/ses/kr.pth\',\n              \'source_sha256\': hashlib.sha256(source_path.read_bytes()).hexdigest(),\n              \'source_shape\': list(source.shape), \'target_shape\': list(target.shape),\n              \'selected_sha256\': hashlib.sha256(selected.read_bytes()).hexdigest()}\n    (work_dir / \'extraction.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    print(\'Official VAD extraction complete; Korean source embedding path/shape/finite values verified.\')\n\n\ndef audio_metrics(path):\n    import numpy as np\n    import soundfile as sf\n    from reference_selection import db\n\n    audio, sr = sf.read(path, dtype=\'float32\', always_2d=True)\n    if not audio.size or not np.isfinite(audio).all():\n        raise ValueError(\'Empty or invalid generated audio\')\n    mono = audio.mean(axis=1)\n    frame = max(1, int(sr * .03))\n    levels = [db(np.sqrt(np.mean(mono[i:i + frame] ** 2))) for i in range(0, len(mono), frame)]\n    result = {\'duration_seconds\': len(audio) / sr, \'sample_rate\': sr,\n              \'peak_dbfs\': db(np.max(np.abs(audio))), \'rms_dbfs\': db(np.sqrt(np.mean(audio ** 2))),\n              \'clipping_fraction\': float(np.mean(np.abs(audio) >= .995)),\n              \'silent_frame_fraction\': float(np.mean(np.array(levels) < -45))}\n    if result[\'rms_dbfs\'] < -45 or result[\'clipping_fraction\'] > .001:\n        raise ValueError(\'Generated audio is silent or clipped; do not export\')\n    return result\n\n\ndef generate_comparisons(checkpoints, work_dir, phrases=(), device=None):\n    from engine import KoreanVoice, QUALITY_MODES\n\n    work_dir = Path(work_dir)\n    extraction = json.loads((work_dir / \'extraction.json\').read_text(encoding=\'utf-8\'))\n    selected = work_dir / \'selected_reference.wav\'\n    if extraction[\'selected_sha256\'] != hashlib.sha256(selected.read_bytes()).hexdigest():\n        raise ValueError(\'Reference selection changed; rerun extraction before generation\')\n    texts = list(dict.fromkeys(normalize_text(t) for t in [*COMPARISON_TEXTS, TEST_TEXT, *phrases]))\n    if len(texts) > 10000:\n        raise ValueError(\'At most 10,000 phrases\')\n    voice = KoreanVoice(checkpoints, work_dir / \'target_se.pth\', device)\n    report = {\'quality_accepted\': False, \'settings\': QUALITY_MODES,\n              \'source_embedding\': str(voice.source_path.name), \'melo_speaker\': \'KR\',\n              \'seed\': 5140, \'samples\': [],\n              \'warning\': \'Signal checks do not certify naturalness, pronunciation, or identity. Listen before export.\'}\n    for mode in QUALITY_MODES:\n        cache = work_dir / mode / \'cache\'\n        cache.mkdir(parents=True, exist_ok=True)\n        for index, text in enumerate(texts):\n            diagnostics = work_dir / \'comparisons\' / mode / f\'{index:02d}\' if index < 3 else None\n            data = validate_wav(voice.synthesize(text, mode, diagnostics=diagnostics))\n            path = cache / cache_name(text)\n            path.write_bytes(data)\n            metrics = audio_metrics(path)\n            if diagnostics:\n                sample = {\'mode\': mode, \'text\': text, \'cloned\': metrics,\n                          \'base\': audio_metrics(diagnostics / \'base.wav\'),\n                          \'self_conversion\': audio_metrics(diagnostics / \'self.wav\')}\n                report[\'samples\'].append(sample)\n            print(f\'Generated {mode} {index + 1}/{len(texts)}\')\n        # Only the current phrase set is eligible for export, never leftover cache.\n    report[\'cache_names\'] = [cache_name(t) for t in texts]\n    report[\'embedding_sha256\'] = hashlib.sha256((work_dir / \'target_se.pth\').read_bytes()).hexdigest()\n    (work_dir / \'quality_report.json\').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    print(\'Comparisons ready. No bundle exported; listen and choose a mode or reject both.\')\n\n\ndef export_bundle(work_dir, output, mode, accepted=False):\n    from engine import QUALITY_MODES\n\n    if not accepted:\n        raise ValueError(\'Export requires explicit listening approval; current/old bundle is not accepted\')\n    work_dir, output = Path(work_dir), Path(output)\n    report = json.loads((work_dir / \'quality_report.json\').read_text(encoding=\'utf-8\'))\n    embedding = work_dir / \'target_se.pth\'\n    if report[\'embedding_sha256\'] != hashlib.sha256(embedding.read_bytes()).hexdigest():\n        raise ValueError(\'Embedding changed; regenerate comparisons first\')\n    extraction = json.loads((work_dir / \'extraction.json\').read_text(encoding=\'utf-8\'))\n    if extraction[\'selected_sha256\'] != hashlib.sha256((work_dir / \'selected_reference.wav\').read_bytes()).hexdigest():\n        raise ValueError(\'Selected reference changed; rerun extraction and comparison\')\n    files = []\n    for name in report[\'cache_names\']:\n        if name != Path(name).name or not name.endswith(\'.wav\'):\n            raise ValueError(\'Invalid generated cache name\')\n        path = work_dir / mode / \'cache\' / name\n        validate_wav(path.read_bytes())\n        audio_metrics(path)\n        files.append(path)\n    manifest = {\'format\': \'manhua-lens-private-v1\', \'private\': True,\n                \'language\': \'KR\', \'openvoice\': OPENVOICE_REV, \'melo\': MELO_REV,\n                \'checkpoints\': CHECKPOINT_REV, \'phrases\': len(files),\n                \'normalization\': \'NFC+strip\', \'quality_mode\': mode,\n                \'settings\': QUALITY_MODES[mode], \'listening_approved\': True,\n                \'extraction\': \'official get_se with VAD on selected 20–30s\'}\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(output, \'w\', zipfile.ZIP_DEFLATED) as bundle:\n        bundle.writestr(\'manifest.json\', json.dumps(manifest))\n        bundle.write(embedding, \'target_se.pth\')\n        for path in files:\n            bundle.write(path, \'cache/\' + path.name)\n    print(\'PRIVATE approved bundle exported; no reference or diagnostics included.\')\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--stage\', choices=[\'select\', \'extract\', \'compare\', \'export\'], required=True)\n    parser.add_argument(\'--reference\', type=Path)\n    parser.add_argument(\'--checkpoints\', type=Path)\n    parser.add_argument(\'--work-dir\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path)\n    parser.add_argument(\'--phrases\', type=Path)\n    parser.add_argument(\'--target-seconds\', type=float, default=25)\n    parser.add_argument(\'--mode\', choices=[\'natural\', \'similarity\'], default=\'natural\')\n    parser.add_argument(\'--accept-quality\', action=\'store_true\')\n    parser.add_argument(\'--device\', choices=[\'cpu\', \'cuda:0\'])\n    args = parser.parse_args()\n    if args.stage == \'select\':\n        if not args.reference:\n            parser.error(\'--reference required for selection\')\n        select_reference(args.reference, args.work_dir, args.target_seconds)\n    elif args.stage == \'extract\':\n        if not args.checkpoints:\n            parser.error(\'--checkpoints required for extraction\')\n        extract_embedding(args.checkpoints, args.work_dir, args.device)\n    elif args.stage == \'compare\':\n        if not args.checkpoints:\n            parser.error(\'--checkpoints required for comparison\')\n        phrases = args.phrases.read_text(encoding=\'utf-8\').splitlines() if args.phrases else []\n        generate_comparisons(args.checkpoints, args.work_dir, [p for p in phrases if p.strip()], args.device)\n    else:\n        if not args.output:\n            parser.error(\'--output required for export\')\n        export_bundle(args.work_dir, args.output, args.mode, args.accept_quality)\n', 'reference_selection.py': '"""Conservative acoustic screening; not diarization or a perceptual quality judge."""\nimport numpy as np\n\n\ndef db(value):\n    return float(20 * np.log10(max(float(value), 1e-8)))\n\n\ndef safe_normalize(audio, target_db=-22.0, peak_db=-3.0):\n    audio = np.asarray(audio, dtype=np.float32)\n    if not audio.size or not np.isfinite(audio).all():\n        raise ValueError(\'Invalid reference samples\')\n    # One constant gain preserves dynamics; no compression, limiting or denoising.\n    audio = audio - np.mean(audio)\n    rms = float(np.sqrt(np.mean(audio ** 2)))\n    peak = float(np.max(np.abs(audio)))\n    if rms < 1e-5:\n        raise ValueError(\'Reference is silent\')\n    gain = min(10 ** (target_db / 20) / rms, 10 ** (peak_db / 20) / peak, 4.0)\n    return audio * gain, {\'gain_db\': db(gain), \'rms_dbfs\': db(rms * gain), \'peak_dbfs\': db(peak * gain)}\n\n\ndef select_segments(audio, sr, intervals, target_seconds=25, pitch_spread=None, clipping_mask=None):\n    """Screen unnormalized VAD speech; rank calm blocks and select 20–30 seconds.\n\n    Pitch spread is an optional callback returning a robust semitone range.\n    Noise estimates use VAD-negative audio. They are proxies, not measured SNR.\n    """\n    if not 20 <= target_seconds <= 30:\n        raise ValueError(\'Target duration must be 20–30 seconds\')\n    audio = np.asarray(audio, dtype=np.float32)\n    if not audio.size or not np.isfinite(audio).all():\n        raise ValueError(\'Invalid audio\')\n    clipped = np.abs(audio) >= .995\n    if clipping_mask is not None:\n        evidence = np.asarray(clipping_mask, dtype=bool)\n        if evidence.shape != audio.shape:\n            raise ValueError(\'Clipping mask must match the audio\')\n        clipped |= evidence\n    mask = np.zeros(len(audio), dtype=bool)\n    bounded = []\n    last_end = 0\n    for start, end in sorted(intervals):\n        start, end = max(last_end, int(start)), min(len(audio), int(end))\n        if end <= start:\n            continue\n        bounded.append((start, end))\n        mask[start:end] = True\n        last_end = end\n    frame = max(1, int(sr * .03))\n    noise_frames = [float(np.sqrt(np.mean(audio[i:i + frame] ** 2)))\n                    for i in range(0, len(audio) - frame + 1, frame)\n                    if not mask[i:i + frame].any()]\n    noise = max(float(np.percentile(noise_frames, 75)), 1e-5) if noise_frames else None\n    candidates, rejected = [], []\n    for start, end in bounded:\n        # Equal blocks avoid a tiny tail; retaining whole VAD spans avoids long blanks.\n        count = max(1, int(np.ceil((end - start) / (5 * sr))))\n        edges = np.linspace(start, end, count + 1, dtype=int)\n        for a, b in zip(edges[:-1], edges[1:]):\n            chunk = audio[a:b]\n            duration = (b - a) / sr\n            rms = float(np.sqrt(np.mean(chunk ** 2)))\n            levels = np.array([db(np.sqrt(np.mean(chunk[i:i + frame] ** 2)))\n                               for i in range(0, len(chunk) - frame + 1, frame)])\n            clipping = float(np.mean(clipped[a:b]))\n            spread = float(np.percentile(levels, 90) - np.percentile(levels, 20)) if levels.size else 99.0\n            snr = db(rms / noise) if noise is not None else None\n            pitch = pitch_spread(chunk, sr) if pitch_spread else None\n            reasons = []\n            if duration < 1.5:\n                reasons.append(\'too_short\')\n            if db(rms) < -38:\n                reasons.append(\'low_volume\')\n            if clipping > .0005:\n                reasons.append(\'clipping\')\n            if snr is not None and snr < 12:\n                reasons.append(\'poor_speech_to_background_proxy\')\n            if spread > 18 or (pitch is not None and pitch > 14):\n                reasons.append(\'large_expression_proxy\')\n            item = {\'start_seconds\': a / sr, \'end_seconds\': b / sr, \'duration\': duration,\n                    \'rms_dbfs\': db(rms), \'clipping_fraction\': clipping,\n                    \'level_spread_db\': spread, \'background_margin_db\': snr,\n                    \'pitch_spread_semitones\': pitch, \'reasons\': reasons}\n            if reasons:\n                rejected.append(item)\n            else:\n                # Prefer lower variation and stronger speech/background margin.\n                score = spread + (pitch or 0) * .5 - min(snr or 20, 40) * .2\n                candidates.append((score, a, b, item))\n    chosen, total = [], 0.0\n    for _, a, b, item in sorted(candidates, key=lambda entry: entry[0]):\n        remaining = target_seconds - total\n        if remaining < 1.5:\n            break\n        if (b - a) / sr > remaining:\n            b = a + int(remaining * sr)\n            item = dict(item, end_seconds=b / sr, duration=(b - a) / sr)\n        chosen.append((a, b, item))\n        total += (b - a) / sr\n    if total < 20:\n        raise ValueError(f\'Only {total:.1f}s passes screening; provide 20–30s of calmer clean speech. Do not relax checks blindly.\')\n    chosen.sort(key=lambda entry: entry[0])\n    # Brief separator prevents discontinuities from turning into false sustained speech.\n    pieces = []\n    for a, b, _ in chosen:\n        chunk = audio[a:b].copy()\n        fade = min(int(sr * .005), len(chunk) // 2)\n        chunk[:fade] *= np.linspace(0, 1, fade)\n        chunk[-fade:] *= np.linspace(1, 0, fade)\n        pieces.extend([chunk, np.zeros(int(sr * .08), dtype=np.float32)])\n    return np.concatenate(pieces[:-1]), {\'selected_speech_seconds\': total,\n        \'selected\': [item for _, _, item in chosen], \'rejected\': rejected,\n        \'candidate_count\': len(candidates), \'noise_estimate_dbfs\': db(noise) if noise is not None else None,\n        \'warning\': \'VAD/acoustics cannot verify one speaker, no music, or perceptual quality. Listen before extraction.\'}\n', 'requirements.txt': '# HTTP layer only. Full model setup: python setup_runtime.py\nfastapi==0.115.6\nuvicorn==0.34.0\n', 'requirements-models.txt': "# Python 3.10; installed by setup_runtime.py into an isolated environment.\n# OpenVoice's numpy==1.22 and ASR dependencies are intentionally bypassed.\n# Melo imports multilingual cleaners eagerly, so their text dependencies remain.\nnumpy==1.26.4\ntorch==2.5.1\ntorchaudio==2.5.1\nlibrosa==0.9.1\nnumba==0.60.0\nsoundfile==0.12.1\ntransformers==4.27.4\nhuggingface-hub==0.25.2\ncached_path==1.6.7\ntxtsplit==1.0.0\nnum2words==0.5.12\nunidic_lite==1.0.8\nunidic==1.1.0\nmecab-python3==1.0.9\npykakasi==2.2.1\nfugashi==1.3.0\ng2p_en==2.1.0\nanyascii==0.3.2\njamo==0.4.1\ngruut[de,es,fr]==2.2.3\ng2pkk==0.1.2\npython-mecab-ko==1.3.7\npydub==0.25.1\neng_to_ipa==0.0.2\ninflect==7.0.0\nunidecode==1.3.7\npypinyin==0.50.0\ncn2an==0.5.22\njieba==0.42.1\nlangid==1.1.6\ntqdm==4.67.1\ntensorboard==2.16.2\nloguru==0.7.2\nnltk==3.8.1\nwavmark==0.0.3\n", 'requirements-extraction.txt': '# Notebook extraction only; never needed by the lightweight local cache/runtime.\n# OpenVoice se_extractor imports these modules even when vad=True (no ASR weights).\nfaster-whisper==1.1.1\nctranslate2==4.5.0\nwhisper-timestamped==1.14.2\nopenai-whisper==20240930\ndtw-python==1.5.3\nonnxruntime==1.20.1\n'}
for name, source in PUBLIC_SOURCES.items():
    (SCRIPTS / name).write_text(source, encoding='utf-8')
SESSION = Path('/tmp/manhua-private-session')
SESSION.mkdir(mode=0o700, exist_ok=True)
RUN = SESSION / 'quality-v2'
RUN.mkdir(mode=0o700, exist_ok=True)
VENV = Path('/tmp/manhua-python310')
PYTHON = str(VENV / 'bin/python')
CHECKPOINTS = Path('/tmp/manhua-checkpoints-v2')
REFERENCE = SESSION / 'openvoice_reference_korean_full.wav'
TORCH_INDEX = 'cu121'  # set 'cpu' without a free GPU


In [ ]:
# OPTIONAL setup/add-on: skip this entire cell after setup succeeds.
# Existing MeloTTS/OpenVoice environment is reused; only extraction imports added once.
if not Path(PYTHON).is_file():
    TOOLS = Path('/tmp/manhua-uv')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(TOOLS), 'uv==0.6.17'], check=True)
    ENV = os.environ.copy()
    ENV['PYTHONPATH'] = str(TOOLS)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.10', '--seed', str(VENV)], env=ENV, check=True)
if not (VENV / 'manhua-ready.json').is_file():
    subprocess.run([PYTHON, str(SCRIPTS / 'setup_runtime.py'), '--torch-index', TORCH_INDEX,
                    '--checkpoints', str(CHECKPOINTS)], check=True)
if not (VENV / 'manhua-extraction-ready.txt').is_file():
    subprocess.run([PYTHON, str(SCRIPTS / 'setup_runtime.py'), '--extraction-only'], check=True)
print('Runtime ready. Future reruns can start at selection, extraction, or comparison.')


## Session-only reference upload
Upload `openvoice_reference_korean_full.wav` (the existing 66-second recording is
an input pool, not all used for embedding). Recommended selected duration: 20–30
seconds of calm, normal Korean speech with one speaker and no music.
If already uploaded in this session, skip the widget/upload cells.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
upload = widgets.FileUpload(accept='.wav', multiple=False)
display(upload)


In [ ]:
REFERENCE = SESSION / 'openvoice_reference_korean_full.wav'
if upload.value:
    # Support ipywidgets 7 (dict) and 8 (tuple).
    items = list(upload.value.values()) if isinstance(upload.value, dict) else list(upload.value)
    assert len(items) == 1
    item = items[0]
    assert item.get('name', item.get('metadata', {}).get('name')) == REFERENCE.name
    assert len(item['content']) <= 128 * 1024 * 1024, 'Reference must be under 128 MiB'
    REFERENCE.write_bytes(bytes(item['content']))
    upload.value = {} if isinstance(upload.value, dict) else ()
    upload.close()
    del items, item
else:
    assert REFERENCE.is_file(), 'Upload the WAV above, or use the Colab fallback cell.'
print('Private reference is in session temporary storage.')


In [ ]:
# OPTIONAL Colab fallback: run only if the widget upload was unavailable.
# from google.colab import files
# previous = Path.cwd()
# os.chdir(SESSION)
# try:
#     uploaded = files.upload()
#     assert list(uploaded) == ['openvoice_reference_korean_full.wav']
#     del uploaded
# finally:
#     os.chdir(previous)
# REFERENCE = SESSION / 'openvoice_reference_korean_full.wav'


In [ ]:
# SELECTION ONLY: rerun to try another reference/target duration. No reinstall.
TARGET_SECONDS = 25
subprocess.run([PYTHON, str(SCRIPTS / 'prepare_voice.py'), '--stage', 'select',
                '--reference', str(REFERENCE), '--work-dir', str(RUN),
                '--target-seconds', str(TARGET_SECONDS)], check=True)


In [ ]:
# Listen to the actual selected reference; acoustic screening is NOT diarization.
import json
from IPython.display import Audio, display
selection = json.loads((RUN / 'selection.json').read_text(encoding='utf-8'))
print(selection)
display(Audio(filename=str(RUN / 'selected_reference.wav')))
# If selection is noisy, contains another voice, or sounds dramatic: reject it.
# Supply a calmer reference and rerun selection; do not blindly lower thresholds.


In [ ]:
# EXTRACTION ONLY: set True only after hearing one calm clean speaker above.
REFERENCE_LISTENED_AND_CLEAN = False
assert REFERENCE_LISTENED_AND_CLEAN, 'Listen to selected reference and confirm it first.'
subprocess.run([PYTHON, str(SCRIPTS / 'prepare_voice.py'), '--stage', 'extract',
                '--checkpoints', str(CHECKPOINTS), '--work-dir', str(RUN)], check=True)
print(json.loads((RUN / 'extraction.json').read_text(encoding='utf-8')))


In [ ]:
# GENERATION ONLY: rerun without extraction/reinstall to compare synthesis settings.
PHRASES = ['한국어', '공부']  # exact additional words/sentences needed locally
phrases_file = RUN / 'phrases.txt'
phrases_file.write_text('\n'.join(PHRASES), encoding='utf-8')
subprocess.run([PYTHON, str(SCRIPTS / 'prepare_voice.py'), '--stage', 'compare',
                '--checkpoints', str(CHECKPOINTS), '--work-dir', str(RUN),
                '--phrases', str(phrases_file)], check=True)


In [ ]:
# Actual comparisons, not merely a successful API response. All previews PRIVATE.
quality = json.loads((RUN / 'quality_report.json').read_text(encoding='utf-8'))
print(quality)
for mode in ['natural', 'similarity']:
    for index, text in enumerate(['안녕하세요.', '오늘은 날씨가 좋아요.', '한국어 공부를 시작해 볼까요?']):
        print(mode, text)
        for kind in ['base', 'self', 'cloned']:
            print(kind)
            display(Audio(filename=str(RUN / 'comparisons' / mode / f'{index:02d}' / (kind + '.wav'))))
# Judge pronunciation, metallic artifacts, pauses/rhythm, and identity separately.
# Base already robotic? MeloTTS limitation. Self conversion metallic? Converter.
# Only cloned version poor? Reference/identity conversion mismatch likely.
# Compare with selected reference at equal playback loudness. Signal statistics
# detect clipping/silence, not naturalness or speaker identity. Reject both if poor.


In [ ]:
# EXPORT ONLY AFTER LISTENING. No bundle is created by extraction/comparison.
CHOSEN_MODE = 'natural'  # or 'similarity'; latter is experimental, not guaranteed better
QUALITY_ACCEPTED_AFTER_LISTENING = False
assert QUALITY_ACCEPTED_AFTER_LISTENING, 'Reject poor results; do not export just because synthesis ran.'
BUNDLE = RUN / ('manhua-private-quality-v2-' + CHOSEN_MODE + '.zip')
assert BUNDLE.name != 'manhua-private.zip', 'Never overwrite the old rejected bundle'
subprocess.run([PYTHON, str(SCRIPTS / 'prepare_voice.py'), '--stage', 'export',
                '--work-dir', str(RUN), '--output', str(BUNDLE), '--mode', CHOSEN_MODE,
                '--accept-quality'], check=True)


In [ ]:
# Download without copying private assets into Kaggle's saved output directory.
# The Kaggle link contains private bytes: clear ALL outputs after downloading.
import base64
from IPython.display import HTML
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(str(BUNDLE))
else:
    encoded = base64.b64encode(BUNDLE.read_bytes()).decode('ascii')
    display(HTML('<a download="' + BUNDLE.name + '" href="data:application/zip;base64,'
                 + encoded + '">Download PRIVATE local assets</a>'))
    del encoded
# If Kaggle/browser blocks the download link, rerun this notebook in free Colab
# and use its native download above. Do not publish assets as a workaround.


## Local use after a genuinely acceptable comparison
The approved ZIP contains only the selected-mode phrase cache, the target speaker
embedding, and a manifest. The raw/selected reference and diagnostics are excluded.
To avoid mixing with rejected cached phrases, use a NEW private asset directory:

```powershell
.\voice_server\start.cmd -Bundle "C:\path\manhua-private-quality-v2-natural.zip" -Assets "C:\path\private-voice-quality-v2"
```

Keep your previous private files untouched. Cache misses still use device Korean TTS.
OpenVoice cannot clone the original reference's emotion/intonation: it transfers
speaker tone color over MeloTTS speech. Do not import either mode if still robotic.

A better free experiment is Chatterbox Multilingual (Korean `ko`), in a separate
private notebook/environment. It is not integrated or claimed superior for this
speaker without an A/B listening test. English-only Chatterbox Turbo/Nano are not
Korean replacements. Official sources:
- https://github.com/myshell-ai/OpenVoice/blob/main/docs/QA.md
- https://github.com/resemble-ai/chatterbox

Download before ending the session, then clear ALL notebook outputs. Do not save
an executed private notebook or use Kaggle saved outputs/datasets as a download workaround.
